In [6]:
from __future__ import annotations

import argparse
import json
from pathlib import Path
from typing import Any

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from pathlib import Path

In [7]:
PROJECT_ROOT = Path.cwd()
DEFAULT_DATASET = PROJECT_ROOT / "Dataset" / "coverwise_synthetic_coverage_risk_dataset_500.csv"
DEFAULT_MODEL = PROJECT_ROOT / "model" / "coverage_risk_model.joblib"

In [8]:

FEATURE_COLUMNS = (
    "policy_type",
    "sum_insured",
    "annual_premium",
    "premium_to_coverage_ratio",
    "waiting_period_months",
    "pre_existing_disease_waiting_period_months",
    "copayment_percentage",
    "room_rent_limit_present",
    "icu_limit_present",
    "disease_sub_limit_count",
    "exclusion_count",
    "restoration_benefit_present",
    "no_claim_bonus_present",
    "child_coverage_present",
    "maternity_coverage_present",
    "claim_process_clarity_score",
    "ambiguous_clause_count",
    "risk_flags_count",
    "coverage_clarity_score",
)
TARGET_COLUMN = "risk_label"
RISK_LABELS = (
    "Strong Coverage",
    "Moderate Coverage",
    "Caution Required",
    "High Risk",
)
NUMERIC_FEATURES = tuple(column for column in FEATURE_COLUMNS if column != "policy_type")
CATEGORICAL_FEATURES = ("policy_type",)

In [9]:
def _label_for_risk(risk_index: float) -> str:
    if risk_index < 25:
        return "Strong Coverage"
    if risk_index < 50:
        return "Moderate Coverage"
    if risk_index < 75:
        return "Caution Required"
    return "High Risk"


In [10]:
def _make_estimators() -> dict[str, Any]:
    return {
        "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=8, class_weight="balanced", random_state=42
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
    }

In [11]:
def _make_preprocessor() -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline((("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler()))),
                list(NUMERIC_FEATURES),
            ),
            (
                "categorical",
                Pipeline(
                    (
                        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
                        ("encode", OneHotEncoder(handle_unknown="ignore")),
                    )
                ),
                list(CATEGORICAL_FEATURES),
            ),
        ]
    )


In [12]:

def train_and_evaluate(
    dataset_path: str | Path = DEFAULT_DATASET,
    model_path: str | Path = DEFAULT_MODEL,
    test_size: float = 0.2,
    random_state: int = 42,
) -> dict[str, Any]:
    """Compare classifiers, validate on a holdout, and save the best full-data pipeline."""
    if not 0 < test_size < 1:
        raise ValueError("test_size must be between 0 and 1.")
    data = pd.read_csv(dataset_path)
    required_columns = {*FEATURE_COLUMNS, TARGET_COLUMN}
    missing = sorted(required_columns.difference(data.columns))
    if missing:
        raise ValueError(f"Dataset is missing required columns: {', '.join(missing)}")
    if data[list(required_columns)].isnull().any().any():
        raise ValueError("Dataset contains missing values in model features or target.")
    unexpected_labels = set(data[TARGET_COLUMN].unique()).difference(RISK_LABELS)
    if unexpected_labels:
        raise ValueError(f"Dataset contains unsupported risk labels: {sorted(unexpected_labels)}")

    x = data.loc[:, FEATURE_COLUMNS]
    y = data[TARGET_COLUMN]
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, test_size=test_size, stratify=y, random_state=random_state
    )

    comparisons: dict[str, dict[str, float]] = {}
    predictions: dict[str, np.ndarray] = {}
    for name, estimator in _make_estimators().items():
        pipeline = Pipeline(
            (("preprocessor", _make_preprocessor()), ("classifier", estimator))
        )
        pipeline.fit(x_train, y_train)
        predicted = pipeline.predict(x_test)
        predictions[name] = predicted
        comparisons[name] = {
            "accuracy": float(accuracy_score(y_test, predicted)),
            "macro_f1": float(f1_score(y_test, predicted, average="macro", zero_division=0)),
        }

    best_name = max(comparisons, key=lambda name: comparisons[name]["macro_f1"])
    final_model = Pipeline(
        (
            ("preprocessor", _make_preprocessor()),
            ("classifier", _make_estimators()[best_name]),
        )
    )
    final_model.fit(x, y)
    destination = Path(model_path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(final_model, destination)

    report = {
        "dataset_rows": int(len(data)),
        "test_rows": int(len(y_test)),
        "class_counts": {str(label): int(count) for label, count in y.value_counts().items()},
        "model_comparison": comparisons,
        "best_validation_model": best_name,
        "best_model_classification_report": classification_report(
            y_test,
            predictions[best_name],
            labels=list(RISK_LABELS),
            output_dict=True,
            zero_division=0,
        ),
        "saved_model": str(destination),
        "disclaimer": (
            "Synthetic educational model only. Predictions are not insurance advice, "
            "actuarial estimates, or a substitute for reviewing policy wording."
        ),
    }
    return report